# Week 10: Agents and tool use

An **agent** is an LLM that can call your functions (**tools**). The LLM decides which tool to call and with what inputs; your Python runs the tool and hands back the result. You'll build one that answers food price questions.

**Before you start:** add the course data with **Add input** (search *akada-labs-course-data*), which includes `nigeria_food_prices.csv`. Add your API key as a secret. On Kaggle: **Add-ons > Secrets**, name it `GROQ_API_KEY` (or `GEMINI_API_KEY`), and switch it on for this notebook. Turn **Internet on** in the settings panel. On Colab: click the key icon on the left and add the same name.

Run each cell with **Shift + Enter**. Every exercise has a check cell that prints **PASS** or a hint.
At the end you get a completion code for the submission form.

In [ ]:
# Run this cell first. You don't need to read it: it holds the self-checks.
EMAIL = ""  # <- the email you applied with

import hashlib, json, os, re

_passed = set()

def ok(n):
    _passed.add(n)
    print(f"PASS: exercise {n} done.")

def hint(message):
    print(f"Not yet. Hint: {message}")

def free_text(n, text, min_sentences):
    parts = [p for p in re.split(r"[.!?]+", text or "") if len(p.split()) >= 3]
    if len(parts) >= min_sentences:
        print("Saved. A TA will read this answer.")
        ok(n)
    else:
        hint(f"write at least {min_sentences} full sentences inside the quotes.")

def completion_code(week, exercises):
    if "@" not in EMAIL:
        return print("Put your email in the first cell, run it, then run this cell again.")
    missing = sorted(set(exercises) - _passed)
    if missing:
        return print(f"Almost there. Not passed yet: {missing}. Run those exercises and checks again.")
    digest = hashlib.sha256(f"akada-labs-cohort-1|{week}|{EMAIL.strip().lower()}".encode()).hexdigest()
    print(f"Well done! Your completion code is: AK-W{week:02d}-{digest[:6].upper()}")
    print("Paste it into the lab submission form.")

def get_secret(name):
    """Read an API key from Kaggle or Colab secrets. Never paste keys into a cell."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    raise RuntimeError(
        f"No secret called {name}. On Kaggle: Add-ons > Secrets. On Colab: the key icon on the left. "
        "Add it there, switch it on for this notebook, then run this cell again.")

print("Ready.")

In [ ]:
!pip install -q openai

In [ ]:
# Pick one provider. Both have a free tier.
PROVIDER = "groq"   # or "gemini"

PROVIDERS = {
    # name: (secret name, OpenAI-compatible endpoint, model)
    "groq":   ("GROQ_API_KEY",   "https://api.groq.com/openai/v1", "llama-3.1-8b-instant"),
    "gemini": ("GEMINI_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.5-flash"),
}
# Model names change. If you get a "model not found" error, check the provider's model list
# (console.groq.com/docs/models or ai.google.dev/gemini-api/docs/models) and update MODEL.

from openai import OpenAI

SECRET_NAME, BASE_URL, MODEL = PROVIDERS[PROVIDER]
client = OpenAI(api_key=get_secret(SECRET_NAME), base_url=BASE_URL)
print("Using", PROVIDER, MODEL)

import ast, operator
import pandas as pd

prices = pd.read_csv("/kaggle/input/akada-labs-course-data/nigeria_food_prices.csv")
print(prices["commodity"].unique(), prices["market"].unique()[:5])
prices.head()

## Exercise 1: Write two tools

Write `get_food_price(commodity, market, month)` that returns the price (a number) for a month like `"2024-06"`, or a message string if there's no match. A `calculator(expression)` that safely does arithmetic is written for you; read how it avoids `eval`.

In [ ]:
def get_food_price(commodity, market, month):
    # filter prices on commodity, market, and date starting with month
    pass

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.USub: operator.neg}

def calculator(expression):
    """Arithmetic only: + - * / and brackets. Never use eval() on text from an LLM."""
    def walk(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](walk(node.left), walk(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](walk(node.operand))
        raise ValueError("only numbers and + - * / are allowed")
    try:
        return walk(ast.parse(expression, mode="eval").body)
    except Exception as error:
        return f"Error: {error}"

row = prices.iloc[0]
print(get_food_price(row["commodity"], row["market"], row["date"][:7]))

In [ ]:
try:
    row = prices.sample(1, random_state=7).iloc[0]
    got = get_food_price(row["commodity"], row["market"], row["date"][:7])
    if not isinstance(got, (int, float)) or abs(got - row["price"]) > 0.01:
        hint("get_food_price should return the price number for that commodity, market and month.")
    elif not isinstance(get_food_price("rice", "Nowhere", "1999-01"), str):
        hint("return a message string when nothing matches, so the agent knows.")
    elif calculator("(1200 - 800) / 800 * 100") != 50 or not str(calculator("__import__('os')")).startswith("Error"):
        hint("calculator was changed; re-run the original cell.")
    else:
        ok(1)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Describe the tools to the LLM

The LLM never sees your Python. It sees a **description** of each tool, in JSON Schema, and replies with
the tool name and arguments it wants. Good descriptions matter as much as good code.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_food_price",
        "description": "Monthly average price in naira of a food item in a Nigerian market.",
        "parameters": {"type": "object", "properties": {
            "commodity": {"type": "string", "enum": sorted(prices["commodity"].unique().tolist())},
            "market": {"type": "string", "enum": sorted(prices["market"].unique().tolist())},
            "month": {"type": "string", "description": "YYYY-MM"},
        }, "required": ["commodity", "market", "month"]},
    }},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression, e.g. '(1500 - 1200) / 1200 * 100'.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
    }},
]
FUNCTIONS = {"get_food_price": get_food_price, "calculator": calculator}

## Exercise 2: The agent loop

Complete `run_agent`. Each step: send the messages to the model; if it asks for tools, run each one and append the result as a `"tool"` message; if it doesn't, return its answer. Fill in the 2 `TODO` lines.

In [ ]:
SYSTEM = "You answer questions about Nigerian food prices. Use the tools for every number; never guess prices."

def run_agent(question, client=client, max_steps=None, log=None):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    step = 0
    while True:
        # (exercise 3 adds the max_steps guard here)
        step += 1
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        message = response.choices[0].message
        if not message.tool_calls:
            return message.content
        messages.append({"role": "assistant", "content": message.content or "", "tool_calls": [
            {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
            for c in message.tool_calls]})
        for call in message.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = None   # TODO: look up the function in FUNCTIONS by call.function.name and call it with **args
            # (exercise 4 adds logging here)
            messages.append(None)   # TODO: {"role": "tool", "tool_call_id": call.id, "content": str(result)}

market = prices["market"].iloc[0]
print(run_agent(f"What was the price of rice in {market} in June 2024, and how much has it changed since June 2022 in percent?"))

In [ ]:
try:
    # The check uses a pretend LLM that asks for the price tool, so it is the same for everyone
    class _Fn:
        def __init__(self, name, arguments): self.name, self.arguments = name, arguments
    class _Call:
        def __init__(self, name, args): self.id, self.type, self.function = "call_1", "function", _Fn(name, json.dumps(args))
    class _Msg:
        def __init__(self, content=None, tool_calls=None): self.role, self.content, self.tool_calls = "assistant", content, tool_calls
    class FakeClient:
        def __init__(self, script): self.script, self.seen = list(script), []
        @property
        def chat(self): return self
        @property
        def completions(self): return self
        def create(self, **kwargs):
            self.seen.append(kwargs["messages"])
            msg = self.script.pop(0) if self.script else _Msg(content="done")
            return type("R", (), {"choices": [type("C", (), {"message": msg})()]})()

    row = prices.iloc[0]
    fake = FakeClient([_Msg(tool_calls=[_Call("get_food_price", {"commodity": row["commodity"], "market": row["market"], "month": row["date"][:7]})]),
                       _Msg(content="final answer")])
    out = run_agent("price?", client=fake)
    tool_msgs = [m for m in fake.seen[-1] if isinstance(m, dict) and m.get("role") == "tool"]
    if out != "final answer" or not tool_msgs or str(round(row["price"], 2))[:4] not in tool_msgs[0]["content"]:
        hint("run the tool from FUNCTIONS and append a tool message with its result, then loop again.")
    else:
        ok(2)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Exercise 3: Stop runaway agents

An agent can get stuck calling tools forever, which costs money and time. Add a guard at the top of the loop: if `max_steps` is set and `step` has reached it, return `"Stopped: too many steps."`. Copy your `run_agent` into this cell and change it.

In [ ]:
# paste your run_agent here and add the guard

In [ ]:
try:
    looping = FakeClient([_Msg(tool_calls=[_Call("calculator", {"expression": "1+1"})]) for _ in range(20)])
    out = run_agent("loop forever", client=looping, max_steps=4)
    if out == "Stopped: too many steps." and len(looping.seen) == 4:
        ok(3)
    else:
        hint(f"with max_steps=4 the model should be called 4 times, then stop. It was called {len(looping.seen)} times.")
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Exercise 4: Log every step

When `log` is a list, append one line per tool call, like `"step 1: get_food_price({...}) -> 1450.0"`. Then print the trace for a real question.

In [ ]:
# paste your run_agent here and add logging

trace = []
print(run_agent(f"Is rice or beans cheaper in {market} in 2024-06?", log=trace, max_steps=6))
print("\n".join(trace))

In [ ]:
try:
    trace_check = []
    fake = FakeClient([_Msg(tool_calls=[_Call("calculator", {"expression": "2*3"})]), _Msg(content="6")])
    run_agent("2*3?", client=fake, log=trace_check, max_steps=5)
    if len(trace_check) == 1 and "calculator" in trace_check[0] and "6" in trace_check[0]:
        ok(4)
    else:
        hint("append one string per tool call to log, including the tool name and its result.")
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Finish

Run the cell below. If every exercise passed, it prints your completion code.

In [ ]:
completion_code(10, [1, 2, 3, 4])